# 支持向量机
支持向量机是一种经典的机器学习方法，在数据量小等条件下其性能仍优于神经网络。

## 1. 引言
如果用苹果的颜色和大小来判断苹果是否成熟，你觉得哪种方法更合适？

1. 通过近邻的几个苹果样本判断。
2. 设定颜色和大小的阈值，当苹果超过该阈值时认定为成熟。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250731093647089.webp" width="400px"/></div>
</div>

显然，上图中红色直线的分类效果比绿色直线的更好。

那么，我们应如何选取一条直线使得分类效果达到最好？这就是支持向量机所探讨的问题。

## 2. 间隔与支持向量


### 2.1 什么是分类？
在机器学习中，**分类**是一种常见的任务，它的目标是根据数据的特征，将数据划分到预定义的类别中。比如，我们想识别一张图片是猫还是狗，或者判断一封邮件是不是垃圾邮件，这些都属于分类问题。

分类问题通常分为两种：

*   **二分类**：只有两个类别，比如“是”或“否”，“猫”或“狗”，“垃圾邮件”或“非垃圾邮件”。SVM 最擅长的就是二分类问题。
*   **多分类**：有三个或更多类别，比如识别手写数字（0-9），或者将新闻文章分为“体育”、“娱乐”、“科技”等类别。虽然 SVM 本身是为二分类设计的，但可以通过一些策略（如“一对多”或“一对一”）来处理多分类问题。

在分类任务中，我们通常会有一个**训练集**，里面包含了已知类别的数据。通过学习这些数据，模型能够找到一个规律，然后用这个规律去预测**新数据**的类别。


### 2.2 线性分类与超平面
有一些数据点，它们分别代表两种不同的东西，比如蓝色小球和红色小球。如果这些小球能够被一条直线（在二维空间中）或者一个平面（在三维空间中）完全分开，那么我们就说这些数据是**线性可分**的。


<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804181658938.gif" width="400px"/></div>
</div>



这条用来分隔不同类别数据的“线”或“面”，我们称之为**超平面（Hyperplane）**。

*   **在二维空间**：超平面就是一条直线。比如，`ax + by + c = 0`。
*   **在三维空间**：超平面就是一个平面。比如，`ax + by + cz + d = 0`。
*   **在更高维空间**：超平面就是一个“广义的平面”，它将整个空间分成两部分，每一部分代表一个类别。

超平面的作用就是作为决策边界。当一个新的数据点出现时，我们只需要看看它落在超平面的哪一侧，就能判断它属于哪个类别。

**超平面的数学表示（简化理解）**

我们可以简单地把它想象成一个函数：

`f(x) = w · x + b`

其中：

*   `x` 代表你的数据点（比如一个蓝色小球的特征，它的x坐标和y坐标）。
*   `w` 是一组权重，它决定了超平面的“方向”或者“倾斜度”。
*   `b` 是一个偏置项，它决定了超平面在空间中的“位置”或者“截距”。

当 `f(x) = 0` 时，`x` 就在超平面上。如果 `f(x)` 大于0，数据点可能属于一类；如果 `f(x)` 小于0，数据点可能属于另一类。SVM的目标就是找到一个最好的 `w` 和 `b`，来定义这个最优的超平面。


### 2.3 间隔与支持向量
如果数据是线性可分的，我们可能会找到很多条直线（超平面）都能把不同类别的数据分开。那么，哪一条才是最好的呢？SVM给出的答案是：**离两边数据都最远的那个超平面，就是最好的！**

这个“离两边数据都最远”的距离，我们称之为**间隔（Margin）**。间隔越大，超平面就越“安全”，在面对新的、未知的数据时，分类的准确性就越高。你可以把它想象成一条高速公路，中间的分隔带越宽，两边的车道就越不容易发生碰撞。

那些距离超平面最近的数据点，它们就像是“卡”在分隔带边缘的车辆，我们称它们为**支持向量（Support Vectors）**。这些支持向量是SVM模型中最重要的点，因为它们直接决定了超平面的位置和间隔的大小。如果这些支持向量的位置发生变化，超平面也会跟着变化；而其他离超平面较远的数据点，即使它们的位置稍微变动，也不会影响超平面。

**为什么支持向量很重要？**

*   **决定超平面**：只有支持向量才对最终的超平面有贡献。非支持向量的数据点，即使被移除，也不会改变超平面。
*   **体现模型复杂度**：支持向量的数量通常比总数据量少得多，这使得SVM在处理高维数据时具有优势，因为它只关注“关键”数据点。

**直观理解间隔最大化**

SVM 的目标就是找到一个超平面，使得它到最近的支持向量的距离最大化。这个距离就是间隔。通过最大化间隔，SVM 能够构建一个鲁棒性更强、泛化能力更好的分类模型。



### 2.4 寻找最优超平面：最大间隔分类器

现在我们知道了，SVM 的目标是找到一个能够最大化间隔的超平面。那么，具体是怎么找到的呢？

我们可以把这个问题想象成一个“优化问题”。我们需要找到超平面的参数（前面提到的 `w` 和 `b`），使得在正确分类所有数据点的同时，超平面到最近的支持向量的距离最大。

**核心思想：**

1.  **正确分类**：首先，超平面必须能够将所有训练数据点正确地划分到它们各自的类别中。这意味着所有蓝色小球都在超平面的一侧，所有红色小球都在超平面的另一侧。
2.  **最大化间隔**：在满足正确分类的前提下，我们希望超平面与距离它最近的那些数据点（支持向量）之间的距离尽可能大。

这个优化过程听起来可能有点复杂，但幸运的是，Scikit-learn 这样的机器学习库已经帮我们封装好了这些复杂的计算。我们只需要调用相应的函数，它就能自动帮我们找到这个最优的超平面。

**为什么最大化间隔很重要？**

*   **更好的泛化能力**：间隔越大，模型对新数据的分类能力就越强。你可以把它看作是模型的“容错能力”，更大的间隔意味着模型在面对稍微有些偏离的数据时，也能做出正确的判断。
*   **更稳定的模型**：如果间隔很小，那么超平面就非常接近某些数据点，这些数据点的小扰动就可能导致超平面发生大的变化，从而影响模型的稳定性。最大化间隔可以使模型更加稳健。

简而言之，最大间隔分类器就是 SVM 的“灵魂”，它确保了模型不仅能正确分类已知数据，还能对未知数据做出更可靠的预测。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.datasets import make_blobs

# 1. 生成模拟数据
# 生成两个类别的数据点
X, y = make_blobs(n_samples=40, centers=2, random_state=6, cluster_std=0.8)

# 2. 训练 SVM 分类器
# 创建一个线性 SVM 分类器
clf = svm.SVC(kernel='linear', C=1000)  # 使用线性核，C 是正则化参数
clf.fit(X, y)  # 训练模型

# 3. 绘制数据点和超平面
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, edgecolors='k')  # 绘制数据点

# 绘制超平面
ax = plt.gca()  # 获取当前坐标轴
xlim = ax.get_xlim()  # 获取 x 轴范围
ylim = ax.get_ylim()  # 获取 y 轴范围

# 创建网格来绘制超平面和间隔边界
x = np.linspace(xlim[0], xlim[1], 30)
y = np.linspace(ylim[0], ylim[1], 30)
Y, X = np.meshgrid(y, x)
xy = np.vstack([X.ravel(), Y.ravel()]).T
P = clf.decision_function(xy).reshape(X.shape)

# 绘制超平面
ax.contour(X, Y, P, colors='k', levels=[-1, 0, 1], alpha=0.5, linestyles=['--', '-', '--'])

# 标记支持向量
ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=100, linewidth=1, facecolors='none', edgecolors='k')

# 设置标题和坐标轴范围
ax.set_xlim(xlim)
ax.set_ylim(ylim)
plt.title('SVM with Linear Kernel')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.show()

上图位于虚线上的三个点即为**支持向量**。

两条虚线中的实线即为所求**分离超平面**，可将两个类别完全分隔开。

## 3. 核函数

### 3.1 线性不可分数据

到目前为止，我们讨论的都是“线性可分”的情况，也就是数据点可以用一条直线（或超平面）完美分开。但现实世界中的数据往往不是那么“规矩”。

如下图，在这种情况下，无论你画多少条直线，都无法将绿色小球和红色小球完全分开。这就是**线性不可分**问题。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804182344180.webp" width="600px"/></div>
</div>

如果我们的 SVM 只能处理线性可分的数据，那它的应用范围就会非常有限。为了解决这个问题，SVM 引入了一个非常巧妙的“魔法”——**核函数（Kernel Function）**。


### 3.2 核技巧的魔力
核函数的核心思想是：**把原始空间中线性不可分的数据，映射到一个更高维度的空间，在这个新的高维空间里，数据就变得线性可分了！**

听起来有点抽象？我们来打个比方：

    想象一下，你有一堆糖果，有些是圆形，有些是方形，它们混在一起，平铺在桌面上，你无法用一把直尺把圆形糖果和方形糖果分开。但是，如果你把这些糖果都捡起来，然后把圆形糖果放在一个高一点的架子上，方形糖果放在低一点的架子上。现在，你就可以用一块平板（相当于三维空间中的一个平面）轻松地把它们分开了。

这里的“捡起来并放到不同高度的架子上”这个动作，就类似于核函数所做的事情。它把数据从低维空间（桌面）“投影”到了高维空间（不同高度的架子），从而使得原本难以分离的数据变得容易分离。

**关键是：** 核函数并没有真正地进行这种高维映射。它很“聪明”，它只是计算了数据点在高维空间中“点积”的结果，而不需要显式地知道高维空间中的坐标。这就好比你不需要真的把糖果放到架子上，只需要知道它们在架子上的相对位置关系，就能判断它们是否能被分开。这种“只计算结果，不进行实际映射”的技巧，就是著名的**核技巧（Kernel Trick）**。

核技巧的优点在于：

*   **避免了高维计算的复杂性**：我们不需要真正地去构建一个高维空间，也不需要计算数据点在高维空间中的具体坐标，这大大降低了计算量。
*   **处理非线性问题**：它使得SVM能够处理各种复杂的非线性分类问题，极大地扩展了SVM的应用范围。



<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804182914570.png" width="600px"/></div>
</div>

这张图展示了支持向量机（SVM）中高斯核（也称为径向基函数核，RBF）的作用，以及如何通过核技巧将数据映射到更高维的特征空间，以便在该空间中找到更好的分类超平面。

**图的组成部分：**

1. **输入空间（Input space）**：
   - 图的左侧显示了原始数据点在二维输入空间中的分布，这里用两个基因（Gene X 和 Gene Y）作为特征。
   - 黑色圆点代表负类对象（y=-1），红色菱形代表正类对象（y=+1）。

2. **特征空间（Feature space）**：
   - 图的右侧显示了通过高斯核映射后的数据点在更高维特征空间中的分布。
   - 高斯核通过映射函数 $\Phi(\vec{x}_i)$ 将原始数据点映射到特征空间。

3. **高斯核（Gaussian kernel）**：
   - 高斯核是一种非线性核函数，它能够将原始数据映射到一个无限维的特征空间中。
   - 在特征空间中，原本在输入空间中线性不可分的数据点变得线性可分。

4. **超平面（Hyperplane）**：
   - 图中右侧的特征空间中，蓝色的平面代表分类超平面，将正类和负类数据点分开。
   - 这个超平面是在特征空间中找到的，而不是在原始的输入空间中。

**高斯核的作用：**

- **非线性映射**：高斯核通过映射函数 $\Phi(\vec{x}_i)$ 将原始数据点映射到一个更高维的空间，使得原本线性不可分的数据点在新的空间中变得线性可分。
- **最大化间隔**：在特征空间中，SVM 寻找一个超平面，该超平面最大化了正类和负类之间的间隔，从而提高了分类器的泛化能力。
- **避免直接计算映射**：核技巧的一个优点是，我们不需要直接计算映射后的高维特征向量，而是通过核函数直接计算映射后的内积，这大大简化了计算。

**总结：**

这张图直观地展示了高斯核如何通过将数据映射到更高维的特征空间来解决非线性分类问题。在特征空间中，SVM 能够找到一个最大化间隔的超平面，从而实现对数据的有效分类。这种方法不仅提高了分类的准确性，还增强了模型对新数据的泛化能力。

下面是一个简单的 Python 代码示例，使用 Scikit-learn 库来展示如何使用 SVM 和核技巧来处理线性不可分的数据。我们将生成一个非线性可分的数据集，并使用高斯核（RBF）来训练 SVM 模型。最后，我们将可视化结果，以展示数据在高维空间中的分类情况。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm
from sklearn.datasets import make_circles

# 生成非线性可分数据
X, y = make_circles(n_samples=100, factor=0.3, noise=0.1, random_state=42)

# 创建 SVM 分类器，使用 RBF 核
clf = svm.SVC(kernel='rbf', C=1.0, gamma=0.1)
clf.fit(X, y)

# 可视化决策边界
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, edgecolors='k', s=50)
plt.title("SVM with RBF Kernel")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")

# 绘制决策边界
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# 创建网格来绘制决策边界
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
Z = clf.decision_function(xy).reshape(XX.shape)

# 绘制决策边界和间隔
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.5, linestyles=['--', '-', '--'])
ax.scatter(clf.support_vectors_[:, 0], clf.support_vectors_[:, 1], s=100, linewidth=1, facecolors='none', edgecolors='k')

plt.show()

运行上述代码后，您将看到一个图形，其中：
- 蓝色和红色的点表示两个类别的数据点。
- 黑色实线表示最优超平面。
- 黑色虚线表示间隔边界。

### 3.3 常用核函数介绍

核函数有很多种，每种核函数都有其特点和适用场景。在实际应用中，最常用的核函数有以下几种：

1.  **线性核（Linear Kernel）**
    *   **作用**：当数据本身就是线性可分的时候，线性核是最佳选择。它相当于没有进行任何高维映射，直接在原始空间中寻找最优超平面。
    *   **特点**：简单、高效，适用于大规模数据集。
    *   **何时使用**：当你确定数据是线性可分，或者特征维度非常高时，可以优先考虑线性核。

2.  **多项式核（Polynomial Kernel）**
    *   **作用**：可以将数据映射到多项式特征空间，从而处理一些非线性问题。
    *   **特点**：通过调整参数（如多项式次数 `d`），可以控制映射的复杂程度。当 `d=1` 时，多项式核就退化为线性核。
    *   **何时使用**：当数据呈现出多项式关系时，可以尝试使用多项式核。

3.  **径向基函数核（Radial Basis Function Kernel, RBF Kernel / Gaussian Kernel）**
    *   **作用**：这是最常用、也是最强大的核函数之一。它能够将原始数据映射到无限维空间，从而处理非常复杂的非线性关系。
    *   **特点**：它基于数据点之间的“距离”来衡量相似性，距离越近，相似性越高。它有一个重要的参数 `gamma`，控制了模型对单个训练样本的影响范围。`gamma` 越大，单个样本的影响范围越小，模型可能越复杂，容易过拟合；`gamma` 越小，单个样本的影响范围越大，模型可能越简单，容易欠拟合。
    *   **何时使用**：在大多数情况下，RBF核都是一个很好的默认选择，因为它能够处理各种复杂的非线性问题。你通常需要通过交叉验证来寻找最佳的 `gamma` 值。

4.  **Sigmoid 核（Sigmoid Kernel）**
    *   **作用**：灵感来源于神经网络中的激活函数，也可以处理非线性问题。
    *   **特点**：在某些情况下表现良好，但不如RBF核通用。
    *   **何时使用**：可以作为 RBF 核的备选，在特定数据集上可能表现出优势。

在实际应用中，选择哪种核函数通常需要通过实验和交叉验证来确定。RBF核由于其强大的非线性处理能力，通常是首选的尝试对象。


## 4. 软间隔

### 4.1 为什么需要软间隔？

我们前面讨论的 SVM，无论是线性的还是通过核函数映射到高维空间的，都假设数据是**完全可分**的。也就是说，总能找到一个超平面，完美地将所有不同类别的数据点分开，并且所有数据点都必须落在间隔带之外。

然而，在现实世界中，数据往往是嘈杂的，可能存在以下情况：

*   **异常点（Outliers）**：有些数据点可能是测量错误、录入错误，或者仅仅是少数的特殊情况，它们的位置与大多数同类数据点格格不入。
*   **数据重叠**：即使是同一类的数据，也可能因为特征相似性、噪声等原因，导致不同类别的数据点在特征空间中有所重叠，无法找到一个完美的超平面将它们完全分开。

如果坚持“硬间隔”（即所有数据点都必须完美分类且落在间隔带外），那么当遇到这些异常点或重叠数据时，SVM可能会：

1.  **找不到合适的超平面**：如果数据中存在无法被完美分开的点，那么“硬间隔”的SVM就无法找到一个满足条件的超平面，导致模型训练失败。
2.  **过拟合**：为了完美地分开所有数据点（包括异常点），模型可能会变得非常复杂，导致对训练数据表现很好，但对新的、未知的数据表现很差，也就是“过拟合”。

为了解决这些问题，SVM引入了**软间隔（Soft Margin）** 的概念。

### 4.2 软间隔的原理

软间隔的核心思想是：**允许一些数据点落在间隔带内，甚至被错误分类，但我们会对这些“犯规”的数据点进行一定的“惩罚”**。

<div class='insertContainerBox column'>
<div class='insertItem' align=center><img src="https://imgbed.momodel.cn/hv/20250804183821283.jpeg" width="600px"/></div>
</div>

为了实现软间隔，我们引入了一个**松弛变量（Slack Variable）**，通常用 $\xi$（读作“克西”）来表示。每个数据点都有一个对应的松弛变量：

*   如果一个数据点被正确分类，并且它离超平面的距离足够远（在间隔带之外），那么它的松弛变量 $\xi$ 就为0。
*   如果一个数据点被正确分类，但它落在了间隔带内，那么它的松弛变量 $\xi$ 就大于 0 但小于 1。
*   如果一个数据点被错误分类，那么它的松弛变量 $\xi$ 就大于1。

SVM 在寻找最优超平面时，除了要最大化间隔之外，还会考虑一个额外的目标：**最小化所有松弛变量的总和**。这意味着我们希望尽可能少地出现“犯规”的数据点，即使有，也希望它们的“犯规程度”尽可能小。

这样，SVM 就不再是“完美主义者”了，它变得更加“宽容”。它允许一些分类错误的存在，以换取更好的泛化能力和对噪声的鲁棒性。

使用 SVM 训练模型，设置 C=0.1 以允许一些误分类。

运行此代码后，我们能看到一个图形，其中用不同的颜色和标记清晰地展示了每个数据点的松弛变量 ξ 的三种情况。

### 4.3 惩罚参数C的理解

既然软间隔允许一些数据点“犯规”，那么我们如何控制这种“犯规”的程度呢？这就需要引入一个非常重要的参数：**惩罚参数 $C$（Cost Parameter）**。

$C$ 是一个正数，它的作用是**平衡“最大化间隔”和“最小化分类错误”这两个目标**。

*   **$C$ 值越大**：
    *   表示我们对分类错误的惩罚越大，模型会更“严格”，更倾向于不容忍任何错误分类的样本。它会努力使所有样本都正确分类，即使这意味着间隔会变小。这可能导致模型变得复杂，容易**过拟合**（对训练数据表现很好，但对新数据表现差）。
    *   你可以想象成一个非常严格的老师，不允许学生犯任何错误，即使是小错误也要严厉惩罚。

*   **$C$ 值越小**：
    *   表示我们对分类错误的惩罚越小，模型会更“宽容”，允许更多的样本落在间隔带内甚至被错误分类。这可能导致模型变得更简单，间隔更大，但可能导致**欠拟合**（模型过于简单，无法捕捉数据中的真实模式）。
    *   你可以想象成一个比较宽容的老师，允许学生犯一些错误，更注重整体的学习效果。

**如何选择合适的 $C$ 值？**

选择一个合适的 $C$ 值非常重要，它直接影响着模型的性能。通常，我们会通过交叉验证（Cross-validation）等方法，尝试不同的 $C$ 值，然后选择在验证集上表现最好的那个。这是一个在“模型复杂度”和“分类错误”之间进行权衡的过程。

通过引入软间隔和惩罚参数 $C$，SVM能够更好地处理那些带有噪声、或者类别之间存在重叠的真实世界数据集，从而提高了模型的鲁棒性和泛化能力。

### 4.4 软间隔 SVM 的简单代码解释
下面我们使用 Python 的 `sklearn` 库来实现一个**软间隔 SVM**，并通过代码解释关键概念（松弛变量 `ξ` 和惩罚参数 `C`）。



#### 4.4.1 生成数据
首先生成一个线性不可分的数据集，模拟真实数据中的噪声或重叠情况：



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from sklearn import svm

# 生成数据（故意制造一些线性不可分的点）
X = np.r_[np.random.randn(20, 2) - [2, 2], np.random.randn(20, 2) + [2, 2]]
y = [0] * 20 + [1] * 20

# 添加一些噪声点（线性不可分）
X = np.r_[X, np.random.randn(5, 2) * 0.5 + [0, 0]]
y += [0, 1, 0, 1, 0]  # 这些点会被错误分类


**解释**：
- 我们生成了 40 个线性可分点（20 个负类 `y=0`，20 个正类 `y=1`）。
- 又添加了 5 个噪声点，使数据变得**线性不可分**（软间隔会允许这些点违规）。

---



#### 4.4.2 训练软间隔 SVM
使用 `sklearn.svm.SVC` 训练模型，并设置 `C=0.1`（允许一定误分类）：






In [ ]:
# 训练软间隔 SVM（C=0.1，允许一定误分类）
clf = svm.SVC(kernel='linear', C=0.1)
clf.fit(X, y)

# 获取支持向量和决策边界
w = clf.coef_[0]
b = clf.intercept_[0]
support_vectors = clf.support_vectors_


**关键参数**：
- `C=0.1`：较小的 `C` 表示允许更多的误分类（模型更“宽容”）。
- `kernel='linear'`：使用线性核（软间隔通常用于线性 SVM）。

---

#### 4.4.3 可视化松弛变量 ξ
松弛变量 `ξ` 衡量样本的违规程度：
- **ξ=0**：样本正确分类且在间隔外。
- **0<ξ<1**：样本正确分类但在间隔内。
- **ξ≥1**：样本被错误分类。





In [ ]:

# 计算每个样本到决策边界的距离
decision_function = clf.decision_function(X)
margin = 1 / np.linalg.norm(w)  # 几何间隔的宽度
distance = decision_function / np.linalg.norm(w)  # 样本到超平面的距离

# 松弛变量 ξ = max(0, 1 - y * (w·x + b))
xi = np.maximum(0, 1 - y * (decision_function + b))  # 注意：这里的 y 应为 ±1（需转换）
y_signed = np.array(y) * 2 - 1  # 将 y 从 [0,1] 转为 [-1,1]
xi = np.maximum(0, 1 - y_signed * decision_function)

# 可视化
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.Paired, edgecolors='k')

# 绘制决策边界和间隔
ax = plt.gca()
xlim = ax.get_xlim()
ylim = ax.get_ylim()

# 创建网格以绘制决策边界
xx = np.linspace(xlim[0], xlim[1], 30)
yy = np.linspace(ylim[0], ylim[1], 30)
YY, XX = np.meshgrid(yy, xx)
xy = np.vstack([XX.ravel(), YY.ravel()]).T
Z = clf.decision_function(xy).reshape(XX.shape)

# 绘制决策边界和间隔
ax.contour(XX, YY, Z, colors='k', levels=[-1, 0, 1], alpha=0.5,
           linestyles=['--', '-', '--'])

# 标记松弛变量 ξ
for i, (xi_val, sv) in enumerate(zip(xi, X)):
    if xi_val > 0:
        plt.scatter(sv[0], sv[1], s=100, facecolors='none', edgecolors='r', linewidths=1.5)
        plt.text(sv[0], sv[1], f'ξ={xi_val:.2f}', fontsize=8, color='red')

plt.title(f"Soft Margin SVM (C={clf.C})")
plt.show()



**输出效果**：
- 决策边界（黑色实线）和间隔（黑色虚线）。
- **红色圆圈**标记了违反间隔的点，并显示其 `ξ` 值：
  - `ξ=0`：无违规（无红圈）。
  - `0<ξ<1`：在间隔内但仍分类正确（红圈较小）。
  - `ξ≥1`：被错误分类（红圈较大）。

---

#### 4.4.4 调整 `C` 的影响
尝试不同的 `C` 值，观察模型变化：
- **`C=0.01`（非常宽松）**：允许更多误分类，间隔更宽。
- **`C=100`（非常严格）**：几乎不允许误分类，间隔更窄（可能过拟合）。


